# GP1 ASR - QuartzNet10x4 on Kaggle

End-to-end pipeline: clone repo -> install extras -> build dev split -> HP search (random, N=20) -> predict -> `/kaggle/working/submission.csv`.

**Prerequisite:** attach the competition dataset `asr-2026-spoken-numbers-recognition-challenge` as **Input** (left panel > Add data). It will mount at `/kaggle/input/asr-2026-spoken-numbers-recognition-challenge/`.

**Recommended accelerator:** P100 or T4 x2. CPU will not finish.

**Internet:** ON (settings > Internet) — needed for `git clone` and `pip install`.

## 1. Clone repository

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git"
REPO_DIR = Path("/kaggle/working/ITMO_Speech_Recognition_Course")
GP1_ROOT = REPO_DIR / "group-projects" / "gp1"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

print("Repo:", REPO_DIR)
print("GP1 root:", GP1_ROOT)
assert (GP1_ROOT / "scripts" / "train.py").exists(), "scripts/train.py not found"

## 2. Install extras only

Kaggle base image ships torch + torchaudio. Installing them again is a waste and risks breaking CUDA. We add only the packages missing from the image.

In [ ]:
!pip install -q num2words pyctcdecode jiwer "pyyaml>=6.0" tqdm soundfile

## 3. Inspect Kaggle input

In [ ]:
DATA_ROOT = Path("/kaggle/input/asr-2026-spoken-numbers-recognition-challenge")
assert DATA_ROOT.exists(), f"Attach the competition dataset first — missing {DATA_ROOT}"

print("DATA_ROOT contents:")
for p in sorted(DATA_ROOT.iterdir()):
    marker = "/" if p.is_dir() else ""
    print(f"  {p.name}{marker}")

## 4. Resolve train / dev / test paths (speaker-aware)

The competition dataset only ships `train/` and `test/`. To validate on **unseen speakers** (which is what Kaggle evaluates), we use one of two strategies:

1. **Preferred**: upload the course-provided `dev/` as a **private Kaggle Dataset** and attach it to this notebook. If present at `/kaggle/input/gp1-dev/`, the cell below uses it directly (same 10-speaker OOD holdout as local/Colab runs).
2. **Fallback**: leave-2-speakers-out split on `train/`. Keeps train and dev speaker-disjoint. Not as OOD as the real dev/, but still measures generalization to unseen speakers.

We **do not** do random row-shuffle on train — that would put the same speakers in both splits and inflate val metric versus test.

In [ ]:
# Prefer an attached private Kaggle dataset with dev/ (recommended).
# Fallback: speaker-aware leave-2-speakers-out split on train.csv.
# Random row-shuffle is explicitly NOT used — it would put the same speakers
# in both splits and make dev CER unrepresentative of test OOD spk.

import csv

DEV_DATASET_ROOT = Path("/kaggle/input/gp1-dev")  # private dataset path

if (DEV_DATASET_ROOT / "dev.csv").exists() and (DEV_DATASET_ROOT / "dev").exists():
    TRAIN_CSV      = DATA_ROOT / "train.csv"
    TRAIN_WAV_ROOT = DATA_ROOT / "train"
    DEV_CSV        = DEV_DATASET_ROOT / "dev.csv"
    DEV_WAV_ROOT   = DEV_DATASET_ROOT / "dev"
    print(f"Using attached private dev dataset: {DEV_DATASET_ROOT}")
else:
    # Speaker-aware fallback (train has 6 speakers; we hold out the last 2 alphabetically).
    SPLIT_DIR = Path("/kaggle/working/splits")
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    TRAIN_CSV      = SPLIT_DIR / "train_split.csv"
    DEV_CSV        = SPLIT_DIR / "dev_split.csv"
    TRAIN_WAV_ROOT = DATA_ROOT / "train"
    DEV_WAV_ROOT   = DATA_ROOT / "train"  # dev wavs live under train/ too

    with open(DATA_ROOT / "train.csv", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    all_speakers = sorted({r["spk_id"] for r in rows})
    holdout = set(all_speakers[-2:])
    train_rows = [r for r in rows if r["spk_id"] not in holdout]
    dev_rows   = [r for r in rows if r["spk_id"] in holdout]

    fieldnames = list(rows[0].keys())
    for path, subset in [(TRAIN_CSV, train_rows), (DEV_CSV, dev_rows)]:
        with open(path, "w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(subset)
    print(f"Speaker-aware fallback split: train={len(train_rows)} ({len(all_speakers)-2} spk), "
          f"dev={len(dev_rows)} (holdout={sorted(holdout)})")

TEST_CSV       = DATA_ROOT / "test.csv"
TEST_WAV_ROOT  = DATA_ROOT / "test"


## HP Search (Random, N=20)

Random search over a 5-dimensional hyperparameter space following Bergstra & Bengio (2012) JMLR — 20 independent trials, each a full training run. The formula N >= log(1-p)/log(1-alpha) guarantees p=0.95 probability of finding a configuration in the top alpha fraction of the search space with N trials.

In [ ]:
import math, json, yaml, sys

BASELINE = "quartznet_10x4"
CONFIG_NAME = "quartznet_10x4.yaml"
N_TRIALS = 20
RANDOM_SEED = 42

OUTPUT_BASE = Path("/kaggle/working/runs/quartznet")
DEVICE = "cuda"
NUM_WORKERS = 2


def sample_params(rng: random.Random) -> dict:
    """5-dim search space per Bergstra & Bengio (2012)."""
    return {
        "lr": 10 ** rng.uniform(-4, math.log10(5e-3)),
        "dropout": rng.uniform(0.05, 0.25),
        "warmup_steps": rng.randint(500, 8000),
        "grad_clip_norm": rng.choice([0.5, 1.0, 2.0, 5.0]),
        "specaug_time_mask_param": rng.choice([15, 25, 35, 50]),
    }


def patch_config(base_path: Path, params: dict, out_path: Path) -> None:
    """Load base config, merge defaults, apply HP params, write patched YAML."""
    with open(base_path) as f:
        cfg = yaml.safe_load(f)
    if "defaults" in cfg:
        for name in cfg.pop("defaults"):
            with open(base_path.parent / f"{name}.yaml") as f:
                base = yaml.safe_load(f)
            merged = {**base}
            for k, v in cfg.items():
                merged[k] = (
                    {**merged[k], **v}
                    if isinstance(v, dict) and isinstance(merged.get(k), dict)
                    else v
                )
            cfg = merged
    cfg.setdefault("train", {})
    cfg["train"]["lr"] = params["lr"]
    cfg["train"].setdefault("optimizer", {})
    cfg["train"]["optimizer"]["lr"] = params["lr"]
    cfg["train"].setdefault("scheduler", {})
    cfg["train"]["scheduler"]["warmup_steps"] = params["warmup_steps"]
    cfg["train"]["grad_clip_norm"] = params["grad_clip_norm"]
    cfg.setdefault("model", {})
    cfg["model"]["dropout"] = params["dropout"]
    cfg.setdefault("aug", {})
    cfg["aug"]["specaug_time_mask_param"] = params["specaug_time_mask_param"]
    with open(out_path, "w") as f:
        yaml.safe_dump(cfg, f)

In [ ]:
import os
rng = random.Random(RANDOM_SEED)
trials_dir = OUTPUT_BASE / "hp_search"
trials_dir.mkdir(parents=True, exist_ok=True)
trials_log = trials_dir / "trials.jsonl"
best = {"trial_id": None, "cer": float("inf"), "ckpt": None, "params": None}

# Ensure tqdm refreshes visibly in Jupyter (non-tty).
os.environ.setdefault("TQDM_MININTERVAL", "0.5")
os.environ.setdefault("PYTHONUNBUFFERED", "1")

for trial_id in range(N_TRIALS):
    params = sample_params(rng)
    trial_dir = trials_dir / f"trial_{trial_id:03d}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    patched = trial_dir / "config.yaml"
    patch_config(GP1_ROOT / "configs" / CONFIG_NAME, params, patched)
    cmd = [
        sys.executable, "-u", "scripts/train.py",
        "--config", str(patched),
        "--train-csv", str(TRAIN_CSV), "--train-root", str(TRAIN_WAV_ROOT),
        "--dev-csv", str(DEV_CSV), "--dev-root", str(DEV_WAV_ROOT),
        "--output-dir", str(trial_dir),
        "--num-workers", str(NUM_WORKERS),
        "--device", DEVICE,
    ]
    print(f"\n===== trial {trial_id:03d}  params={params} =====")

    stderr_log_path = trial_dir / "stderr.log"
    import subprocess
    proc = subprocess.Popen(
        cmd,
        cwd=str(GP1_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )

    # Forward stdout line-by-line (progress bars, INFO logs).
    stderr_tail: list[str] = []
    with open(stderr_log_path, "w") as stderr_fh:
        import threading

        def _pipe_stderr():
            for line in proc.stderr:
                stderr_fh.write(line)
                stderr_fh.flush()
                stderr_tail.append(line)
                if len(stderr_tail) > 200:
                    stderr_tail.pop(0)
                # Mirror WARN/ERROR lines live to keep user informed.
                if " ERROR " in line or " WARNING " in line or "Traceback" in line:
                    print(line, end="")

        t = threading.Thread(target=_pipe_stderr, daemon=True)
        t.start()
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        t.join(timeout=2.0)

    if proc.returncode != 0:
        print(f"\ntrial {trial_id:03d}: FAILED (rc={proc.returncode})")
        print("--- stderr (last 25 lines) ---")
        print("".join(stderr_tail[-25:]))
        cer = float("inf")
        result = {"error": f"rc={proc.returncode}"}
    else:
        try:
            result = json.loads((trial_dir / "result.json").read_text())
            cer = float(result["best_val_cer"])
        except Exception as exc:
            print(f"\ntrial {trial_id:03d}: could not parse result.json: {exc}")
            cer = float("inf")
            result = {"error": str(exc)}

    with open(trials_log, "a") as f:
        f.write(json.dumps({"trial_id": trial_id, "params": params, "best_val_cer": cer}) + "\n")
    if cer < best["cer"]:
        best = {
            "trial_id": trial_id,
            "cer": cer,
            "ckpt": result.get("best_ckpt_path") if isinstance(result, dict) else None,
            "params": params,
        }
    print(f"trial {trial_id:03d}: CER={cer:.4f}  lr={params['lr']:.2e}")

print(f"\nBEST trial_id={best['trial_id']} CER={best['cer']:.4f}  ckpt={best['ckpt']}")
(trials_dir / "best_trial.json").write_text(json.dumps(best, indent=2, default=str))


## 5. Predict and write submission.csv

In [ ]:
SUBMISSION_CSV = Path("/kaggle/working/submission.csv")
cmd = [
    sys.executable, "scripts/predict.py",
    "--checkpoint", best["ckpt"],
    "--config", str(trials_dir / f"trial_{best['trial_id']:03d}" / "config.yaml"),
    "--test-csv", str(TEST_CSV), "--test-root", str(TEST_WAV_ROOT),
    "--output", str(SUBMISSION_CSV),
    "--device", DEVICE,
]
print("$", " ".join(cmd))
subprocess.run(cmd, cwd=str(GP1_ROOT), check=True)
print("Wrote:", SUBMISSION_CSV)

## 6. Preview submission

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/submission.csv")
print(df.shape)
df.head(10)